## Cholesky Decomposition
行權價格 K、到期時間 T、無風險利率 r、模擬次數 ns、重複次數 nr、資產數 n、初始資產價格 S、股息率 q、波動率 sigma、相關係數 rho


In [1]:
import numpy as np

In [9]:
def cholesky_decomposition(K, T, r, ns, nr, n, S, q, sigma, rho):
    # miu = ln(S0*exp((r-q-sigma**2/2)*T))
    miu = []
    for i in range(n):
        miui = np.log(S) + (r - q - sigma ** 2 / 2) * T
        miu.append(miui)
    #     print(miu)
    payoff_mean = []
    for repeat in range(nr):
        # Covariance Matrix
        C = np.zeros([n, n])
        for i in range(n):
            for j in range(n):
                if i == j:
                    C[i][j] = sigma ** 2 * T
                if i < j:
                    C[i][j] = rho * sigma * sigma * T
                else:
                    C[i][j] = C[j][i]
        #         print(C)
        # Cholesky decomposition
        U = np.zeros([n, n])  
        sum_akikj = 0
        for i in range(n):
            for j in range(i, n):       
                sum_akikj = sum(U[k][i] * U[k][j] for k in range(i))
                if i == j:
                    U[i][j] = (C[i][i] - sum_akikj) ** 0.5
                else:
                    U[i][j] = 1.0 / U[i][i] * (C[i][j] - sum_akikj)
        #         print(U)
        # put N(0,1)into Z matrix
        z = np.zeros([ns, n])
        for i in range(ns):
            for j in range(n):
                z[i][j] = np.random.normal()
        rj = z.dot(U)  # z*U = r
        # ST = e^r
        S_list = np.zeros([ns, n])
        for i in range(ns):
            for j in range(n):
                rj[i][j] += miu[j]
                S_list[i][j] = np.exp(rj[i][j])
        #         print(S_list)
        # rainbow option
        payoff_list = []
        for i in range(ns):
            max_payoff = 0
            for j in range(n):
                payoff = max(S_list[i][j] - K, 0)
                if max(S_list[i][j] - K, 0) > max_payoff:
                    max_payoff = max(S_list[i][j] - K, 0)
            discount_payoff = np.exp(-r * T) * max_payoff
            payoff_list.append(discount_payoff)
        payoff_mean.append(np.mean(payoff_list))
    price = np.mean(payoff_mean)
    price_se = np.std(payoff_mean)
    price_down = price - 2 * price_se
    price_up = price + 2 * price_se
    print(f"Rainbow maximum call: {price}")
    print(f"95%C.I:[{price_up}, {price_down}]")
    print(f"Confidence interval width: {price_up - price_down}")

In [27]:
K = 100
T = 0.5
r = 0.1
ns = 10000
nr = 20
n = 2
S = 95
q = 0.05
sigma = 0.5
rho = 1

cholesky_decomposition(K, T, r, ns, nr, n, S, q, sigma, rho)
print("==================================================")
rho = -1
cholesky_decomposition(K, T, r, ns, nr, n, S, q, sigma, rho)
print("==================================================")
n = 5
rho = 0.5
cholesky_decomposition(K, T, r, ns, nr, n, S, q, sigma, rho)


Rainbow maximum call: 11.965339463530075
95%C.I:[12.396662220733463, 11.534016706326687]
Confidence interval width: 0.8626455144067755
Rainbow maximum call: 23.951665141639847
95%C.I:[24.612091828303498, 23.291238454976195]
Confidence interval width: 1.320853373327303
Rainbow maximum call: 30.27060661739703
95%C.I:[30.918214262277587, 29.62299897251647]
Confidence interval width: 1.295215289761117


## Antithetic Variate Approach and Moment Matching Method

In [11]:
def cholesky_decomposition_avammm(K, T, r, ns, nr, n, S, q, sigma, rho):
    miu = []
    for i in range(n):
        miui = np.log(S) + (r - q - sigma ** 2 / 2) * T
        miu.append(miui)
    #     print(miu)
    price_mean = []
    for repeat in range(nr):
        # Covariance Matrix
        C = np.zeros([n, n])
        for i in range(n):
            for j in range(n):
                if i == j:
                    C[i][j] = sigma ** 2 * T
                if i < j:
                    C[i][j] = rho * sigma * sigma * T
                else:
                    C[i][j] = C[j][i]
        #         print(C)
        # Cholesky decomposition
        U = np.zeros([n, n]) 
        sum_akikj = 0
        for i in range(n):
            for j in range(i, n):
                sum_akikj = sum(U[k][i] * U[k][j] for k in range(i))
                if i == j:
                    U[i][j] = (C[i][i] - sum_akikj) ** 0.5
                else:
                    U[i][j] = 1.0 / U[i][i] * (C[i][j] - sum_akikj)
        #         print(U)
        # antithetic variate approach
        z = np.zeros([ns, n])
        half_ns = round(ns/2)
        for i in range(ns):
            for j in range(n):
                if i <= ns / 2:
                    z[i][j] = np.random.normal()
                else:
                    z[i][j] = -z[i - half_ns][j]
        # moment matching method
        zj_std_list = []
        for j in range(n):
            zj_list = []
            for i in range(ns):
                zj_list.append(z[i][j])
            zj_std = np.std(zj_list)
            zj_std_list.append(zj_std)

        y = np.zeros([ns, n])
        for i in range(ns):
            for j in range(n):
                y[i][j] = z[i][j] / zj_std_list[j]
        rj = y.dot(U)

        S_list = np.zeros([ns, n])
        for i in range(ns):
            for j in range(n):
                rj[i][j] += miu[j]
                S_list[i][j] = np.exp(rj[i][j])
        #         print(S_list)

        payoff_list = []
        for i in range(ns):
            max_payoff = 0
            for j in range(n):
                payoff = max(S_list[i][j] - K, 0)
                if max(S_list[i][j] - K, 0) > max_payoff:
                    max_payoff = max(S_list[i][j] - K, 0)
            discount_payoff = np.exp(-r * T) * max_payoff
            payoff_list.append(discount_payoff)
        price_mean.append(np.mean(payoff_list))
    price = np.mean(price_mean)
    price_se = np.std(price_mean)
    price_down = price - 2 * price_se
    price_up = price + 2 * price_se
    print(f"Rainbow maximum call: {price}")
    print(f"95%C.I:[{price_up}, {price_down}]")
    print(f"Confidence interval width: {price_up - price_down}")

In [28]:
K = 100
T = 0.5
r = 0.1
ns = 10000
nr = 20
n = 2
S = 95
q = 0.05
sigma = 0.5
rho = 1

cholesky_decomposition_avammm(K, T, r, ns, nr, n, S, q, sigma, rho)
print("==================================================")
rho = -1
cholesky_decomposition_avammm(K, T, r, ns, nr, n, S, q, sigma, rho)
print("==================================================")
n = 5
rho = 0.5
cholesky_decomposition_avammm(K, T, r, ns, nr, n, S, q, sigma, rho)

Rainbow maximum call: 11.970375012192477
95%C.I:[12.047311711529671, 11.893438312855283]
Confidence interval width: 0.15387339867438854
Rainbow maximum call: 23.954363995219744
95%C.I:[24.08903164957883, 23.81969634086066]
Confidence interval width: 0.26933530871816913
Rainbow maximum call: 30.347946795016206
95%C.I:[30.53339956054983, 30.16249402948258]
Confidence interval width: 0.3709055310672511


## Inverse Cholesky

https://homepage.ntu.edu.tw/~jryanwang/papers/2008,%20Variance%20Reduction%20for%20Multivariate%20Monte%20Carlo%20Simulation%20(Wang).pdf
(P.4)

In [25]:
import numpy as np

# http://homepage.ntu.edu.tw/~jryanwang/
# Variance Reduction for Multivariate Monte Carlo Simulation
def inverse_cholesky(K, T, r, ns, nr, n, S, q, sigma, rho):
    miu = []
    for i in range(n):
        miui = np.log(S) + (r - q - sigma ** 2 / 2) * T
        miu.append(miui)
    #     print(miu)
    payoff_mean = []
    for repeat in range(nr):
        # Step 1: Generate independent standard normal distributed random samples for each underlying asset and obtain a matrix of random samples
        z = np.zeros([ns, n])
        half_ns = round(ns/2)
        for i in range(ns):
            for j in range(n):
                if i <= ns / 2:
                    z[i][j] = np.random.normal()
                else:
                    z[i][j] = -z[i - half_ns][j]              
        # Step 2: Calculate the variance-covariance matrix C tilde
        zj_miu_list = []  # Get miu_zj
        for j in range(n):
            zj_list = []
            for i in range(ns):
                zj_list.append(z[i][j])
            zj_miu = np.mean(zj_list)
            zj_miu_list.append(zj_miu)
        # z_tilde = zj - miuj
        z_tilde = np.zeros([ns, n])
        for i in range(ns):
            for j in range(n):
                z_tilde[i][j] = z[i][j] - zj_miu_list[j]
        # Covariance matrix C based on z
        z_tilde_transpose = np.array(z_tilde).T
        C_tilde = np.cov(z_tilde_transpose)
        # Step 3: Based on the covariance matrix C tilde, performthe Cholesky decomposition
        U_tilde = np.zeros([n, n])  
        sum_akikj = 0
        for i in range(n):
            for j in range(i, n):
                sum_akikj = sum(U_tilde[k][i] * U_tilde[k][j] for k in range(i))
                if i == j:
                    U_tilde[i][j] = (C_tilde[i][i] - sum_akikj) ** 0.5
                else:
                    U_tilde[i][j] = 1.0 / U_tilde[i][i] * (C_tilde[i][j] - sum_akikj)
        # Step 4: Get Zj'
        U_inverse = np.linalg.inv(U_tilde)
        z_prime = z_tilde.dot(U_inverse)

        C = np.zeros([n, n])
        for i in range(n):
            for j in range(n):
                if i == j:
                    C[i][j] = sigma ** 2 * T
                if i < j:
                    C[i][j] = rho * sigma * sigma * T
                else:
                    C[i][j] = C[j][i]
        #         print(C)
        # Cholesky decomposition
        U = np.zeros([n, n])  # U: upper上三角矩陣
        sum_akikj = 0
        for i in range(n):
            for j in range(i, n):
                sum_akikj = sum(U[k][i] * U[k][j] for k in range(i))
                if i == j:
                    U[i][j] = (C[i][i] - sum_akikj) ** 0.5
                else:
                    U[i][j] = 1.0 / U[i][i] * (C[i][j] - sum_akikj)

        rj = z_prime.dot(U)
        # r取exp即為ST
        S_list = np.zeros([ns, n])
        for i in range(ns):
            for j in range(n):
                rj[i][j] += miu[j]
                S_list[i][j] = np.exp(rj[i][j])
        #         print(S_list)
        # rainbow option
        payoff_list = []
        for i in range(ns):
            max_payoff = 0
            for j in range(n):
                payoff = max(S_list[i][j] - K, 0)
                if max(S_list[i][j] - K, 0) > max_payoff:
                    max_payoff = max(S_list[i][j] - K, 0)
            discount_payoff = np.exp(-r * T) * max_payoff
            payoff_list.append(discount_payoff)
        payoff_mean.append(np.mean(payoff_list))
    price = np.mean(payoff_mean)
    price_se = np.std(payoff_mean)
    price_down = price - 2 * price_se
    price_up = price + 2 * price_se
    print(f"Rainbow maximum call: {price}")
    print(f"95%C.I:[{price_up}, {price_down}]")
    print(f"Confidence interval width: {price_up - price_down}")

In [29]:
K = 100
T = 0.5
r = 0.1
ns = 10000
nr = 20
n = 2
S = 95
q = 0.05
sigma = 0.5
rho = 1

inverse_cholesky(K, T, r, ns, nr, n, S, q, sigma, rho)
print("==================================================")
rho = -1
inverse_cholesky(K, T, r, ns, nr, n, S, q, sigma, rho)
print("==================================================")
n = 5
rho = 0.5
inverse_cholesky(K, T, r, ns, nr, n, S, q, sigma, rho)






Rainbow maximum call: 11.970695162303098
95%C.I:[12.020348957667279, 11.921041366938917]
Confidence interval width: 0.09930759072836182
Rainbow maximum call: 23.948196416743794
95%C.I:[24.088477708498946, 23.807915124988643]
Confidence interval width: 0.28056258351030294
Rainbow maximum call: 30.383143551760178
95%C.I:[30.546442074929388, 30.219845028590967]
Confidence interval width: 0.3265970463384207
